Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
#Read all pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """process all pdf files in a directory"""
    all_documents =  []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to the metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

In [6]:
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: test.pdf
Loaded 1 pages

Processing: test1.pdf
Loaded 2 pages

 Total documents loaded: 3


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_tag': '10, 0, 1, 1', 'author': 'Davis Boban', 'moddate': '2025-10-14T12:30:02+05:30', 'source': '..\\data\\pdf\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf', 'file_type': 'pdf'}, page_content='Ag

In [9]:
#Text splitting get into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {(len(documents))} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks = split_documents(all_pdf_documents)
chunks

Split 3 documents into 5 chunks

Example chunk:
Content: Agentic AI refers to artificial intelligence systems that exhibit agency—meaning they 
can autonomously pursue goals, make decisions, and take actions in dynamic 
environments, often with minimal huma...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_tag': '10, 0, 1, 1', 'author': 'Davis Boban', 'moddate': '2025-10-14T12:30:02+05:30', 'source': '..\\data\\pdf\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf', 'file_type': 'pdf'}, page_content='Ag